In [1]:
import sys
import os
import subprocess
import numpy as np
import pandas as pd
import matplotlib
import sklearn
import scipy
import sys
import torch
import gymnasium as gym
import ale_py

result = subprocess.run(["sw_vers"], capture_output=True, text=True)
print(result.stdout.strip())
print(f"Python executable {sys.executable}")
print(f"Python:       {sys.version}")
print(f"NumPy:        {np.__version__}")
print(f"Pandas:       {pd.__version__}")
print(f"Matplotlib:   {matplotlib.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"SciPy:        {scipy.__version__}")
print(f"Torch:        {torch.__version__}")
print(f"Gymnasium:    {gym.__version__}")

# Check available hardware
if torch.cuda.is_available():
    print(f'GPU is available, using GPU {torch.cuda.get_device_name(0)}')
    device = torch.device('cuda')
elif torch.mps.is_available():
    print(f'Apple Metal Performance Shaders framework is available, using {torch.device("mps")}')
    device = torch.device('mps')
else:
    print('Hardware acceleration not available, using CPU')
    device = torch.device('cpu')


ProductName:		macOS
ProductVersion:		15.7.4
BuildVersion:		24G517
Python executable /Users/mjserena/Documents/GitHub/DATA442_DL/.venv/bin/python
Python:       3.14.8 (v3.14.8:8e6e75d9102, Sep 30 2026, 14:34:30) [Clang 21.0.0 (clang-2100.1.1.101)]
NumPy:        2.5.3
Pandas:       3.0.6
Matplotlib:   3.11.2
Scikit-learn: 1.9.1
SciPy:        1.18.1
Torch:        2.14.1
Gymnasium:    1.3.0
Apple Metal Performance Shaders framework is available, using mps


In [2]:
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torchinfo import summary

In [3]:
# MsPacmanDQN is a neural network that maps states → Q-values.

# This network receives:
# a stack of 4 game frames (shape (4, 84, 84)),
# 4 is the number of stacked frames
# 84×84 is the downsampled grayscale image (standard DQN pre-processing)

# and outputs:
# a vector of Q-values, one for each possible action.

class MsPacmanDQN(nn.Module):
    """
    Deep Q-Network (DQN) for Atari Ms. Pac-Man.
    
    Architecture based on the classic DeepMind DQN:
    
    Input:
        4 stacked grayscale frames, each 84×84 pixels.
    
    Convolutional layers:
        conv1: 4 → 32 channels, 8×8 kernel, stride 4
        conv2: 32 → 64 channels, 4×4 kernel, stride 2
        conv3: 64 → 64 channels, 3×3 kernel, stride 1
    
        These layers progressively extract increasingly complex
        spatial features while reducing spatial dimensions
        from 84×84 to 7×7.
    
    Fully connected layers:
        fc1: 3136 → 512
        out: 512 → num_actions
    
    Output:
        One Q-value for each available discrete action.
    
    The CNN learns spatial representations that the DQN uses
    to estimate the expected discounted future reward of
    each possible action.
    """

    def __init__(self, num_actions, in_channels=4):
        super().__init__()
        # Input: (batch, in_channels=4, 84, 84)

        # | Layer | Input → Output   | Kernel | Stride | Purpose                        |
        # | ----- | ---------------- | ------ | ------ | ------------------------------ |
        # | conv1 | 4 → 32 channels  | 8×8    | 4      | Capture large spatial patterns |
        # | conv2 | 32 → 64 channels | 4×4    | 2      | Detect medium patterns         |
        # | conv3 | 64 → 64 channels | 3×3    | 1      | Fine details                   |
        # These layers reduce the image spatial size from 84×84 → 7×7 (exactly as in DQN).

        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)


        # ---------------------------------------------------------
        # Fully Connected Layers (DQN "head")
        # ---------------------------------------------------------
        # Flatten 64 feature maps of size 7×7 = 3136 units.
        #
        # fc1:
        #   - Hidden layer with 512 units
        #   - Provides a dense, high-level representation of the scene.
        #
        # out:
        #   - Final layer, size = number of actions in the environment
        #   - Each output neuron is Q(s, a) for one specific action.
        # ---------------------------------------------------------

        # 84x84 → after convs this will be 7x7 if you follow the usual Atari setup
        # After the conv layers, the tensor is flattened:
        # 64 channels * 7 * 7 = 3136 numbers
        # This adds a fully-connected hidden layer with 512 units
        self.fc1 = nn.Linear(64 * 7 * 7, 512)
        self.out = nn.Linear(512, num_actions)


    def forward(self, x):
        # x expected as uint8 in [0,255] if coming straight from env
        # Normalize pixels
        # Input coming from Atari is uint8 in [0,255]
        # Dividing by 255 turns them into [0,1]
        # Three convolutional layers with ReLU
        # Flatten
        # Fully connected layer (512 units)
        # Output layer - Produces one Q-value per action

        # Normalize raw pixel values from [0, 255] → [0, 1]
        x = x / 255.0

        # Apply convolutional layers with ReLU activations
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))

        # Flatten all feature maps into a single vector per sample
        x = x.view(x.size(0), -1)

        # Fully connected layers (dense representation → Q-values)
        x = F.relu(self.fc1(x))
        return self.out(x)  # Q-values for all actions


In [10]:
# qnet = “online network”
# The CNN that:
# sees every new state during training
# computes Q-values to pick actions
# gets updated every training step by gradient descent
# learns from replay buffer samples
# is constantly changing as it learns
num_actions = 9
qnet = MsPacmanDQN(num_actions, in_channels=4).to(device)

In [11]:
print("CCN Model Structure:\n")

for name, layer in qnet.named_children():
    print(f"({name}): {layer}")

    for param_name, param in layer.named_parameters():
        print(f"    {param_name}: {param.size()}")

print()

CCN Model Structure:

(conv1): Conv2d(4, 32, kernel_size=(8, 8), stride=(4, 4))
    weight: torch.Size([32, 4, 8, 8])
    bias: torch.Size([32])
(conv2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
    weight: torch.Size([64, 32, 4, 4])
    bias: torch.Size([64])
(conv3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    weight: torch.Size([64, 64, 3, 3])
    bias: torch.Size([64])
(fc1): Linear(in_features=3136, out_features=512, bias=True)
    weight: torch.Size([512, 3136])
    bias: torch.Size([512])
(out): Linear(in_features=512, out_features=9, bias=True)
    weight: torch.Size([9, 512])
    bias: torch.Size([9])



In [12]:
model = MsPacmanDQN(num_actions, in_channels=4).to(device)
model_stats = summary(
    model,
    input_size=(1, 4, 84, 84),
    device=device,
    col_names=("input_size", "output_size", "num_params", "kernel_size"),
    depth=2,
    verbose=0
)

with open("cnn_baseline_summary.txt", "w") as f:
    f.write(str(model_stats))

print(model_stats)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Kernel Shape
MsPacmanDQN                              [1, 4, 84, 84]            [1, 9]                    --                        --
├─Conv2d: 1-1                            [1, 4, 84, 84]            [1, 32, 20, 20]           8,224                     [8, 8]
├─Conv2d: 1-2                            [1, 32, 20, 20]           [1, 64, 9, 9]             32,832                    [4, 4]
├─Conv2d: 1-3                            [1, 64, 9, 9]             [1, 64, 7, 7]             36,928                    [3, 3]
├─Linear: 1-4                            [1, 3136]                 [1, 512]                  1,606,144                 --
├─Linear: 1-5                            [1, 512]                  [1, 9]                    4,617                     --
Total params: 1,688,745
Trainable params: 1,688,745
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 9.37
Input si

### Potential CNN Experiments

| Experiment | Architectural Change | What It Investigates |
|------------|----------------------|----------------------|
| Baseline | Original CNN, FC1 = 512 | Reference performance |
| 1 | Reduce FC1 to 256 | Effect of fewer parameters |
| 2 | Reduce FC1 to 128 | Whether a much smaller network performs comparably |
| 3 | Increase convolutional filters | Effect of greater feature extraction capacity |
| 4 | Add a fourth convolutional layer | Effect of network depth |
| 5 | Change kernel sizes | Effect of spatial receptive fields |